# Permuting and sampling data to calculate the null hypothesis

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
data_loc = '../data/server_ready'

files = [os.path.join(data_loc, f) for f in os.listdir(data_loc) if (not f.startswith('._')) and f.endswith('.csv') and ('null-corpus' not in f)]
print(files)

['../data/server_ready/cha_corpus.csv', '../data/server_ready/cha_corpus-2.csv', '../data/server_ready/cha_corpus-3.csv', '../data/server_ready/grace_corpus.csv', '../data/server_ready/cha_corpus-4.csv', '../data/server_ready/cha_corpus-coraal.csv']


In [3]:
null_output_file = 'null-corpora/{}-null-corpus_permuted-within-convo.tsv'
null_output_file = os.path.join(data_loc, null_output_file)
null_output_file

'../data/server_ready/null-corpora/{}-null-corpus_permuted-within-convo.tsv'

## Setting up the null-test data

### Load, permute, save

In [5]:
for f in tqdm(files):
    print(f)
    df = pd.read_csv(f)

    # print(df[['x_utterance', 'y_utterance']].isna().sum(),'\n')
    df = df.loc[
        (~df['x_utterance'].isna())
        & (~df['y_utterance'].isna())
    ]
    # print(df[['x_utterance', 'y_utterance']].isna().sum())

    df['x_turn_id'] = df['conversation_id'].astype(str) + '-' + df['x_turn_id'].astype(str)

    df['sample_delta'] = df.groupby(by=['x_turn_id', 'self']).cumcount() + 1


    print('finding examples')
    # a lot of work just to find useful examples ...
    good_examples = df[['corpus', 'x_turn_id', 'self', 'sample_delta']].groupby(by=['corpus', 'x_turn_id', 'self', 'sample_delta']).agg('max').reset_index()
    good_examples = pd.merge(
        left=good_examples.loc[good_examples['self']], left_on='x_turn_id',
        right=good_examples[['x_turn_id', 'self', 'sample_delta']].loc[~good_examples['self']], right_on='x_turn_id',
        how='left'
    )
    good_examples = good_examples.loc[
        ((good_examples['sample_delta_x'] >= 5)
        & (good_examples['sample_delta_x'] <= 70))
        & ((good_examples['sample_delta_y'] >= 5)
        & (good_examples['sample_delta_y'] <= 70))
    ]

    df = df.loc[df['x_turn_id'].isin(good_examples['x_turn_id'])]
    # print(df[['x_utterance', 'y_utterance']].isna().sum(), '\n')

    print('permuting sentences')
    for cid in tqdm(df['conversation_id'].unique()):
        sub = df.loc[df['conversation_id'].isin([cid])]
        df['y_utterance'].loc[sub.index] = sub['y_utterance'].sample(frac=1.0)

    print('saving data')
    for c in tqdm(df['corpus'].unique()):

        sub = df['x_turn_id'].loc[df['corpus'].isin([c])].unique()

        k = int(np.ceil(len(sub)*.1))

        if k > 0:
            sub = np.random.choice(sub, size=(k,), replace=False)
            df_ = df.loc[df['x_turn_id'].isin(sub)]

            print('\n', c, k, df_.shape, '\n', df_[['x_utterance', 'y_utterance']].isna().sum())

            null_output_file_ = str(null_output_file).format(c)
            # save subset to null output file.
            # if os.path.exists(null_output_file):
            #     df_.to_csv(null_output_file, index=False, header=False, encoding='utf-8', mode='a', sep='\t')
            # else:
            df_.to_csv(null_output_file_, index=False, encoding='utf-8', sep='\t')
    print('===][===')

  0%|          | 0/6 [00:00<?, ?it/s]

../data/server_ready/cha_corpus.csv
finding examples
permuting sentences


  0%|          | 0/240 [00:00<?, ?it/s]

saving data


  0%|          | 0/7 [00:00<?, ?it/s]

...


 MICASE-advising 409 (25062, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-colloquia 773 (44060, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-defense 334 (19855, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-discussion 603 (33794, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-interviews 177 (10601, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-labs 1000 (60107, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-lel 1002 (49989, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-les 2078 (114266, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-meetings 664 (39343, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-office_hours 1977 (118423, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-seminars 1146 (67988, 20) 
 x_utterance    0
y_utterance    0
dtype: int64

 MICASE-service_encounters 457 (26833, 20) 
 x_utterance    0
y_utterance    0
dtype:

  0%|          | 0/210 [00:00<?, ?it/s]

saving data


  0%|          | 0/16 [00:00<?, ?it/s]


 se0-ag1 1504 (108480, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se0-ag2 1374 (138894, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se0-ag3 1283 (135243, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se0-ag4 585 (54489, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se1-ag1 1429 (94822, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se1-ag2 874 (52562, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se1-ag3 913 (64464, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se1-ag4 681 (40288, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se2-ag1 1096 (79580, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se2-ag2 645 (39220, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se2-ag3 1038 (69257, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se2-ag4 1192 (74549, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se3-ag1 1205 (91352, 19) 
 x_utterance    0
y_utterance    0
dtype: int64

 se3-ag2 501 

### test that file works

In [6]:
corpora_are_in = os.path.join(data_loc, 'null-corpora')
corpora_are_in

'../data/server_ready/null-corpora'

In [7]:
corpora = [f for f in os.listdir(corpora_are_in) if (not f.startswith('._')) and f.endswith('.tsv')]
corpora

['callfriend-eng_n-null-corpus_permuted-within-convo.tsv',
 'callfriend-eng_s-null-corpus_permuted-within-convo.tsv',
 'callhome-null-corpus_permuted-within-convo.tsv',
 'instruction-DISPEL-null-corpus_permuted-within-convo.tsv',
 'instruction-Frederiksen-null-corpus_permuted-within-convo.tsv',
 'instruction-Graesser-null-corpus_permuted-within-convo.tsv',
 'instruction-med_school-null-corpus_permuted-within-convo.tsv',
 'GCSAusE-null-corpus_permuted-within-convo.tsv',
 'SBCSAE-null-corpus_permuted-within-convo.tsv',
 'SCoSE-null-corpus_permuted-within-convo.tsv',
 'CABNC-0missing-null-corpus_permuted-within-convo.tsv',
 'CABNC-KB0-null-corpus_permuted-within-convo.tsv',
 'CABNC-KB1-null-corpus_permuted-within-convo.tsv',
 'CABNC-KB2-null-corpus_permuted-within-convo.tsv',
 'CABNC-KB3-null-corpus_permuted-within-convo.tsv',
 'CABNC-KB4-null-corpus_permuted-within-convo.tsv',
 'CABNC-KB5-null-corpus_permuted-within-convo.tsv',
 'CABNC-KB6-null-corpus_permuted-within-convo.tsv',
 'CABNC-

In [8]:
df = []

In [9]:
for f in corpora:
    df += [pd.read_table(os.path.join(corpora_are_in,f), sep='\t')]
    # print(f, df[-1].shape)
    # print(df[-1][['x_utterance', 'y_utterance']].isna().sum())
    # print('===][===\n')
df = pd.concat(df, ignore_index=True)

In [10]:
df[['x_utterance', 'y_utterance']].isna().sum()

x_utterance    0
y_utterance    0
dtype: int64

In [12]:
null_output_file = os.path.join(data_loc,'null-corpus_permuted-within-convo.csv')

In [13]:
df.to_csv(null_output_file, index=False, encoding='utf-8')

#### Test that stitched doc is okay

In [14]:
null_output_file = os.path.join(data_loc,'null-corpus_permuted-within-convo.csv')

In [15]:
df = pd.read_csv(null_output_file)
# df = pd.read_table(null_output_file, sep='\t')
df.shape

(4114590, 32)

In [16]:
df[['x_utterance', 'y_utterance']].isna().sum()

x_utterance    0
y_utterance    0
dtype: int64

In [17]:
df.head()

,document_line_no,x_turn_id,x_speaker,x_utterance,bullet,recipient,conversation_id,overlapping_text,corpus,com,...,y_utterance_delta_no,self,sample_delta,wor,x_timestamp,x_speaker_loc,x_speaker_gender,y_timestamp,y_speaker_loc,y_speaker_gender
0,55.0,4175-22,callfriend-eng_n-4175-M2,noise,"[46411, 46731]",ADULT,4175,True,callfriend-eng_n,NaN,...,0.0,False,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,55.0,4175-22,callfriend-eng_n-4175-M2,noise,"[46411, 46731]",ADULT,4175,True,callfriend-eng_n,NaN,...,1.0,False,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,55.0,4175-22,callfriend-eng_n-4175-M2,noise,"[46411, 46731]",ADULT,4175,True,callfriend-eng_n,NaN,...,2.0,False,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,55.0,4175-22,callfriend-eng_n-4175-M2,noise,"[46411, 46731]",ADULT,4175,True,callfriend-eng_n,NaN,...,3.0,False,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,55.0,4175-22,callfriend-eng_n-4175-M2,noise,"[46411, 46731]",ADULT,4175,True,callfriend-eng_n,NaN,...,4.0,False,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
